# 2회차 실습: Motion Planning Essentials for the Final Project

이 노트북은 학생 배포용 실습 자료입니다.

이번 시간에는 final project에 바로 필요한 trajectory 기초만 다룹니다.

이번 노트북에서 제공되는 것:
- 2R 로봇 모델과 시각화 코드
- trajectory 결과를 바로 볼 수 있는 plotting / animation 셀
- joint-space / Cartesian-space 비교 예제

이번 노트북에서 직접 구현할 것:
- `cubic_coefficients()`
- `estimate_via_velocities()`
- `interpolate_position_linear()`

실습 방법:
1. 위에서부터 셀을 순서대로 실행합니다.
2. `TODO(student)`가 있는 함수만 직접 구현합니다.
3. 구현 후 바로 아래 그래프와 애니메이션으로 결과를 확인합니다.



이번 실습에서 다룰 내용은 아래와 같습니다.

1. 시작 자세와 목표 자세를 cubic joint trajectory로 부드럽게 연결하기
2. via point를 넣어서 경로를 의도적으로 꺾기
3. joint-space 계획과 Cartesian-space 계획이 어떻게 다른지 보기

즉, 이 노트북은 motion planning의 전체 그림을 빠르게 잡기 위한 내용입니다.


## 0. Imports and Utility Functions


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
try:
    from IPython.display import HTML, display
except ImportError:
    def HTML(x):
        return x

    def display(x):
        return x

np.set_printoptions(precision=4, suppress=True)


def wrap_to_pi(angle):
    return (angle + np.pi) % (2 * np.pi) - np.pi


def wrap_vec(q):
    return wrap_to_pi(np.asarray(q, dtype=float))


def rad2deg(q):
    return np.rad2deg(q)


def deg2rad(q):
    return np.deg2rad(q)


## 1. 2R Robot Model

이번 실습에서도 같은 2R 로봇을 사용합니다.
이번에는 FK와 IK 자체보다, `시간에 따라 관절각이 어떻게 바뀌는가`에 더 집중합니다.

trajectory를 비교할 때는 보통 두 가지를 같이 봅니다.

- joint profile: 각 관절의 위치, 속도, 가속도
- end-effector path: 말단이 작업공간에서 실제로 지나가는 경로

아래 모델은 그 두 가지를 모두 확인하기 위해 필요한 것들이 구현된 class이다.


In [ ]:
class Planar2R:
    def __init__(self, link_lengths=(1.0, 0.8), joint_limits=None):
        self.l = np.array(link_lengths, dtype=float)
        if joint_limits is None:
            self.joint_limits = np.array([
                [-np.pi, np.pi],
                [-np.pi, np.pi],
            ], dtype=float)
        else:
            self.joint_limits = np.array(joint_limits, dtype=float)

    def fk(self, q):
        q = np.asarray(q, dtype=float)
        q1, q2 = q
        l1, l2 = self.l
        q12 = q1 + q2
        x = l1 * np.cos(q1) + l2 * np.cos(q12)
        y = l1 * np.sin(q1) + l2 * np.sin(q12)
        alpha = wrap_to_pi(q12)
        return np.array([x, y, alpha])

    def position(self, q):
        return self.fk(q)[:2]

    def joint_positions(self, q):
        q = np.asarray(q, dtype=float)
        q1, q2 = q
        l1, l2 = self.l
        p0 = np.array([0.0, 0.0])
        p1 = p0 + l1 * np.array([np.cos(q1), np.sin(q1)])
        p2 = p1 + l2 * np.array([np.cos(q1 + q2), np.sin(q1 + q2)])
        return np.vstack([p0, p1, p2])

    def ik_analytic(self, target_position, elbow='up', check_limits=True):
        x, y = np.asarray(target_position, dtype=float)
        l1, l2 = self.l
        r2 = x**2 + y**2
        c2 = (r2 - l1**2 - l2**2) / (2 * l1 * l2)
        if c2 < -1.0 - 1e-9 or c2 > 1.0 + 1e-9:
            return None
        c2 = np.clip(c2, -1.0, 1.0)
        s2_abs = np.sqrt(max(0.0, 1.0 - c2**2))
        s2 = s2_abs if elbow == 'up' else -s2_abs
        q2 = np.arctan2(s2, c2)
        k1 = l1 + l2 * c2
        k2 = l2 * s2
        q1 = np.arctan2(y, x) - np.arctan2(k2, k1)
        q = wrap_vec([q1, q2])
        if check_limits and not self.within_limits(q):
            return None
        return q

    def ik_all(self, target_position, check_limits=True):
        sols = []
        for elbow in ['up', 'down']:
            q = self.ik_analytic(target_position, elbow=elbow, check_limits=check_limits)
            if q is not None:
                sols.append((elbow, q))
        return sols

    def within_limits(self, q):
        q = wrap_vec(q)
        return np.all(q >= self.joint_limits[:, 0]) and np.all(q <= self.joint_limits[:, 1])

    def plot(self, q, ax=None, target_position=None, title=None, show_frame=True, color='C0'):
        if ax is None:
            fig, ax = plt.subplots(figsize=(6, 6))
        pts = self.joint_positions(q)
        ax.plot(pts[:, 0], pts[:, 1], '-o', linewidth=4, markersize=8, color=color)
        ax.scatter([0], [0], s=80, marker='s', color='k', label='base')
        ee = pts[-1]
        alpha = self.fk(q)[2]
        if show_frame:
            scale = 0.18
            ax.arrow(ee[0], ee[1], scale * np.cos(alpha), scale * np.sin(alpha), head_width=0.04, length_includes_head=True, color='r')
        if target_position is not None:
            x, y = target_position
            ax.scatter([x], [y], s=120, marker='*', color='red', label='target')
        reach = np.sum(self.l)
        ax.set_xlim(-reach - 0.2, reach + 0.2)
        ax.set_ylim(-reach - 0.2, reach + 0.2)
        ax.set_aspect('equal')
        ax.grid(True)
        if title:
            ax.set_title(title)
        return ax


robot = Planar2R(link_lengths=(1.0, 0.8))
q_demo = deg2rad([25, 50])
print('q_demo [deg] =', rad2deg(q_demo))
print('FK [x, y, alpha(rad)] =', robot.fk(q_demo))
robot.plot(q_demo, title='2R Planar Robot');


## 2. Cubic Joint-Space Trajectory

가장 먼저 볼 것은 가장 기본적인 부드러운 trajectory입니다.

시작 자세 `q0`와 목표 자세 `qf`가 주어졌을 때,
각 joint를 cubic으로 연결하면 시작과 끝에서 속도를 0으로 만들 수 있습니다.

이번 프로젝트 baseline에서도 이런 식의 `joint-space smooth motion`이 기본 뼈대가 됩니다.


### 구현 포인트: Cubic 계수 함수

이 셀에서는 각 joint에 대해

$
q(t) = a_0 + a_1 t + a_2 t^2 + a_3 t^3
$

형태의 계수를 계산합니다.

입력은 시작/끝 자세와 시간 `T`이고,
출력은 각 joint의 계수 배열입니다.



In [ ]:
def cubic_coefficients(q0, qf, T, v0=None, vf=None):
    q0 = np.asarray(q0, dtype=float)
    qf = np.asarray(qf, dtype=float)
    if v0 is None:
        v0 = np.zeros_like(q0)
    if vf is None:
        vf = np.zeros_like(q0)
    v0 = np.asarray(v0, dtype=float)
    vf = np.asarray(vf, dtype=float)

    # TODO(student): cubic 계수 a0, a1, a2, a3를 계산해 보세요.
    raise NotImplementedError('Implement cubic_coefficients() before running the next cells.')


def evaluate_cubic(coeffs, t):
    a0, a1, a2, a3 = coeffs
    t = np.asarray(t, dtype=float)
    q = a0[None, :] + a1[None, :] * t[:, None] + a2[None, :] * t[:, None]**2 + a3[None, :] * t[:, None]**3
    qd = a1[None, :] + 2 * a2[None, :] * t[:, None] + 3 * a3[None, :] * t[:, None]**2
    qdd = 2 * a2[None, :] + 6 * a3[None, :] * t[:, None]
    return q, qd, qdd


### Example: 한 구간을 부드럽게 연결하기

숫자와 그래프를 모두 볼 필요는 없습니다.
우선 아래 두 가지만 확인하면 됩니다.

- 시작 속도와 끝 속도가 0에 가깝다
- joint-space에서 부드럽게 움직이면 end-effector 경로는 직선이 아닐 수 있다


In [ ]:
q0 = deg2rad([20, 40])
qf = deg2rad([100, -30])
T = 3.0
N = 151

t = np.linspace(0.0, T, N)
coeffs_cubic = cubic_coefficients(q0, qf, T)
q_cubic, qd_cubic, qdd_cubic = evaluate_cubic(coeffs_cubic, t)

print('q0 [deg] =', rad2deg(q0))
print('qf [deg] =', rad2deg(qf))
print('start velocity [deg/s] =', rad2deg(qd_cubic[0]))
print('final velocity [deg/s] =', rad2deg(qd_cubic[-1]))


In [ ]:
def plot_joint_profiles(t, q, qd, qdd, title_prefix='Trajectory'):
    n = q.shape[1]
    labels = [rf'$q_{i + 1}$' for i in range(n)]
    fig, axes = plt.subplots(3, 1, figsize=(8, 8), sharex=True)
    for i in range(n):
        axes[0].plot(t, rad2deg(q[:, i]), label=labels[i])
        axes[1].plot(t, rad2deg(qd[:, i]), label=labels[i])
        axes[2].plot(t, rad2deg(qdd[:, i]), label=labels[i])
    axes[0].set_ylabel('Position [deg]')
    axes[1].set_ylabel('Velocity [deg/s]')
    axes[2].set_ylabel('Acceleration [deg/s²]')
    axes[2].set_xlabel('Time [s]')
    axes[0].set_title(title_prefix)
    for ax in axes:
        ax.grid(True)
        ax.legend(loc='best')
    plt.tight_layout()


plot_joint_profiles(t, q_cubic, qd_cubic, qdd_cubic, title_prefix='Cubic Joint-space Trajectory')


In [ ]:
def compute_ee_positions(robot, qs):
    return np.array([robot.position(q) for q in qs])


x_cubic = compute_ee_positions(robot, q_cubic)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(x_cubic[:, 0], x_cubic[:, 1], linewidth=2, label='end-effector path')
robot.plot(q_cubic[0], ax=ax, color='C0', title='End-effector Path from Joint-space Cubic')
robot.plot(q_cubic[-1], ax=ax, color='C1')
ax.scatter(x_cubic[0, 0], x_cubic[0, 1], marker='*', s=200, color='C2', label='start')
ax.scatter(x_cubic[-1, 0], x_cubic[-1, 1], marker='*', s=200, color='C3', label='goal')
ax.legend()


In [ ]:
# functions for gif animation
# don't modify this block!!!

def animate_robot_motion(robot, qs, interval=50, stride=2, title='Robot Motion'):
    qs = np.asarray(qs, dtype=float)
    qs_vis = qs[::stride]
    reach = np.sum(robot.l) + 0.2

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_xlim(-reach, reach)
    ax.set_ylim(-reach, reach)
    ax.set_aspect('equal')
    ax.grid(True)
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('y')

    line, = ax.plot([], [], '-o', linewidth=3, markersize=7, color='C0')
    trace, = ax.plot([], [], '-', linewidth=1.5, alpha=0.7, color='C1')
    trace_pts = []

    def init():
        line.set_data([], [])
        trace.set_data([], [])
        return line, trace

    def update(frame):
        pts = robot.joint_positions(qs_vis[frame])
        line.set_data(pts[:, 0], pts[:, 1])
        trace_pts.append(pts[-1])
        arr = np.array(trace_pts)
        trace.set_data(arr[:, 0], arr[:, 1])
        return line, trace

    anim = FuncAnimation(fig, update, frames=len(qs_vis), init_func=init, interval=interval, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())


def animate_trajectory_comparison(robot, qs_a, qs_b, label_a='joint-space', label_b='Cartesian-space', interval=50, stride=2):
    qs_a = np.asarray(qs_a, dtype=float)[::stride]
    qs_b = np.asarray(qs_b, dtype=float)[::stride]
    n = min(len(qs_a), len(qs_b))
    qs_a = qs_a[:n]
    qs_b = qs_b[:n]
    reach = np.sum(robot.l) + 0.2

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_xlim(-reach, reach)
    ax.set_ylim(-reach, reach)
    ax.set_aspect('equal')
    ax.grid(True)
    ax.set_title('Joint-space vs Cartesian-space Motion')
    ax.set_xlabel('x')
    ax.set_ylabel('y')

    line_a, = ax.plot([], [], '-o', linewidth=3, markersize=6, color='C0', label=label_a)
    line_b, = ax.plot([], [], '-o', linewidth=3, markersize=6, color='C3', label=label_b)
    trace_a, = ax.plot([], [], '-', linewidth=1.2, alpha=0.7, color='C0')
    trace_b, = ax.plot([], [], '-', linewidth=1.2, alpha=0.7, color='C3')
    trace_pts_a, trace_pts_b = [], []
    ax.legend(loc='upper right')

    def init():
        line_a.set_data([], [])
        line_b.set_data([], [])
        trace_a.set_data([], [])
        trace_b.set_data([], [])
        return line_a, line_b, trace_a, trace_b

    def update(frame):
        pts_a = robot.joint_positions(qs_a[frame])
        pts_b = robot.joint_positions(qs_b[frame])
        line_a.set_data(pts_a[:, 0], pts_a[:, 1])
        line_b.set_data(pts_b[:, 0], pts_b[:, 1])
        trace_pts_a.append(pts_a[-1])
        trace_pts_b.append(pts_b[-1])
        arr_a = np.array(trace_pts_a)
        arr_b = np.array(trace_pts_b)
        trace_a.set_data(arr_a[:, 0], arr_a[:, 1])
        trace_b.set_data(arr_b[:, 0], arr_b[:, 1])
        return line_a, line_b, trace_a, trace_b

    anim = FuncAnimation(fig, update, frames=n, init_func=init, interval=interval, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())


### 움직이는 GiF로 확인

아래 애니메이션에서는 같은 cubic trajectory를 시간 순서대로 따라갑니다. 
시작과 끝에서 부드럽게 출발하고 멈추는 것을 시각적으로 확인하세요.

In [ ]:
animate_robot_motion(robot, q_cubic, title='Cubic Joint-space Motion')


## 3. Via Point로 경로 바꾸기

이제 중간 자세(via point)를 하나 이상 넣어서 경로를 바꿔 보겠습니다.

이러한 것들은 trajectory를 계획할 때, obstacle을 피해 우회하는 상황 등 원하는 motion을 만들기 위해 사용된다.

이번 실습에서는 중간 via point의 속도를 적당히 추정해서, 멈추지 않고 자연스럽게 통과하는 piecewise cubic trajectory를 만듭니다.


### 구현 포인트: Via Point 속도 추정하기

이 셀에서는 중간 via point의 속도를 정합니다.

핵심은 아래와 같다.

- 양옆 구간의 기울기를 보고, 비슷한 방향이면 평균을 쓰고
- 방향이 바뀌면 0으로 둔다

이 규칙은 단순하지만, 학생들이 처음 trajectory를 설계해 보기에는 충분히 직관적입니다.


In [ ]:
def estimate_via_velocities(q_points, times):
    q_points = np.asarray(q_points, dtype=float)
    times = np.asarray(times, dtype=float)
    K, n = q_points.shape
    v = np.zeros((K, n))

    # TODO(student): via point 속도 추정 규칙을 구현해 보세요.
    raise NotImplementedError('Implement estimate_via_velocities() before running the next cells.')


def piecewise_cubic_with_via_velocities(q_points, segment_times, samples_per_segment=80):
    q_points = np.asarray(q_points, dtype=float)
    segment_times = np.asarray(segment_times, dtype=float)
    times = np.concatenate([[0.0], np.cumsum(segment_times)])
    velocities = estimate_via_velocities(q_points, times)

    all_t, all_q, all_qd, all_qdd = [], [], [], []
    time_offset = 0.0
    for k, Tk in enumerate(segment_times):
        local_t = np.linspace(0.0, Tk, samples_per_segment, endpoint=(k == len(segment_times) - 1))
        coeffs = cubic_coefficients(q_points[k], q_points[k + 1], Tk, v0=velocities[k], vf=velocities[k + 1])
        q, qd, qdd = evaluate_cubic(coeffs, local_t)
        if k > 0:
            local_t = local_t[1:]
            q, qd, qdd = q[1:], qd[1:], qdd[1:]
        all_t.append(local_t + time_offset)
        all_q.append(q)
        all_qd.append(qd)
        all_qdd.append(qdd)
        time_offset += Tk

    return np.concatenate(all_t), np.vstack(all_q), np.vstack(all_qd), np.vstack(all_qdd), velocities


q_via = np.array([
    deg2rad([20, 40]),
    deg2rad([60, 80]),
    deg2rad([100, 10]),
    deg2rad([70, -50]),
])
segment_times = [1.5, 1.2, 1.7]

t_smooth, q_smooth, qd_smooth, qdd_smooth, via_vel = piecewise_cubic_with_via_velocities(q_via, segment_times, samples_per_segment=90)

print('Estimated via velocities [deg/s]:')
print(rad2deg(via_vel))
plot_joint_profiles(t_smooth, q_smooth, qd_smooth, qdd_smooth, title_prefix='Piecewise Cubic Trajectory with Via Points')


In [ ]:
x_smooth = compute_ee_positions(robot, q_smooth)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(t_smooth, rad2deg(qd_smooth[:, 0]), label='q1 velocity')
axes[0].set_title('Velocity through via points')
axes[0].set_xlabel('Time [s]')
axes[0].set_ylabel('Velocity [deg/s]')
axes[0].grid(True)
axes[0].legend()

axes[1].plot(x_smooth[:, 0], x_smooth[:, 1], linewidth=2, label='end-effector path')
for i, q in enumerate(q_via):
    pos = robot.position(q)
    axes[1].scatter(pos[0], pos[1], s=100, label=f'via {i}')
axes[1].set_aspect('equal')
axes[1].set_title('Path shaped by via points')
axes[1].grid(True)
axes[1].legend()
plt.tight_layout()


### Via Point Motion Animation

이번에는 via point가 들어간 trajectory를 실제 움직임으로 봅니다. 경로가 어떻게 꺾이고, 중간점들을 자연스럽게 통과하는지 확인해 보세요.


In [ ]:
animate_robot_motion(robot, q_smooth, title='Motion with Via Points')


## 4. Joint-Space Trajectory vs Cartesian-Space Trajectory

마지막으로, 같은 시작점과 목표점이라도 trajectory를 어디에서 만들었는지에 따라 결과가 달라진다는 점만 확인하겠습니다.

- joint-space trajectory: 관절각을 직접 부드럽게 보간
- Cartesian-space trajectory: 말단 위치를 직선으로 보간한 뒤, 매 점마다 IK를 풀어 관절각으로 바꿈

이 비교는 final project에서 `그냥 joint trajectory로 갈지`, `waypoint를 EE 기준으로 설계할지`를 판단하는 기준이 됩니다.


### 구현 포인트: Cartesian 경로를 joint trajectory로 바꾸기

이 셀에서는 다음 순서를 코드로 실행합니다.

1. 시작/끝 자세를 정한다.
2. joint-space trajectory를 하나 만든다.
3. Cartesian 직선 경로도 하나 만든다.
4. Cartesian 경로의 각 점마다 IK를 풀어 joint trajectory로 바꾼다.

즉, Cartesian-space planning은 매 시점마다 IK가 필요하다는 점이 핵심입니다.


In [ ]:
def choose_closest_solution(robot, target_position, q_ref, check_limits=True):
    sols = robot.ik_all(target_position, check_limits=check_limits)
    if len(sols) == 0:
        return None
    best_q = None
    best_cost = np.inf
    for _, q in sols:
        diff = wrap_vec(q - q_ref)
        cost = np.linalg.norm(diff)
        if cost < best_cost:
            best_cost = cost
            best_q = q
    return best_q


def interpolate_position_linear(p0, pf, s_values):
    p0 = np.asarray(p0, dtype=float)
    pf = np.asarray(pf, dtype=float)

    # TODO(student): path parameter s를 이용해 선형 position interpolation을 구현해 보세요.
    raise NotImplementedError('Implement interpolate_position_linear() before running the next cells.')


q_start = deg2rad([25, 50])
q_goal = deg2rad([115, -35])
pos_start = robot.position(q_start)
pos_goal = robot.position(q_goal)

T_cart = 3.0
t_cart = np.linspace(0.0, T_cart, 150)
s = t_cart / T_cart
s = 3 * s**2 - 2 * s**3

q_joint, qd_joint, qdd_joint = evaluate_cubic(cubic_coefficients(q_start, q_goal, T_cart), t_cart)
pos_joint = compute_ee_positions(robot, q_joint)

pos_cart_des = interpolate_position_linear(pos_start, pos_goal, s)
q_cart = []
q_ref = q_start.copy()
failed_idx = None
for i, pos_des in enumerate(pos_cart_des):
    q_sol = choose_closest_solution(robot, pos_des, q_ref, check_limits=True)
    if q_sol is None:
        failed_idx = i
        break
    q_cart.append(q_sol)
    q_ref = q_sol
q_cart = np.array(q_cart)
pos_cart_actual = compute_ee_positions(robot, q_cart) if len(q_cart) > 0 else np.empty((0, 2))

print('start position =', pos_start)
print('goal position  =', pos_goal)
print('Cartesian IK failed index:', failed_idx)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(pos_joint[:, 0], pos_joint[:, 1], linewidth=2, label='joint-space trajectory')
ax.plot(pos_cart_des[:, 0], pos_cart_des[:, 1], '--', linewidth=2, label='desired Cartesian straight path')
if len(pos_cart_actual) > 0:
    ax.plot(pos_cart_actual[:, 0], pos_cart_actual[:, 1], linewidth=2, label='IK-tracked Cartesian trajectory')

robot.plot(q_start, ax=ax, color='C0', title='Joint-space vs Cartesian-space Trajectory')
robot.plot(q_goal, ax=ax, color='C1')
ax.scatter(pos_start[0], pos_start[1], s=100, marker='o', label='start')
ax.scatter(pos_goal[0], pos_goal[1], s=140, marker='*', label='goal')
ax.legend();


### 두 방식의 움직임을 같이 보기

아래 애니메이션은 joint-space trajectory와 Cartesian-space trajectory를 동시에 보여 줍니다. 같은 시작점과 목표점이어도 중간 motion이 다르게 생긴다는 점이 더 직관적으로 보입니다.


In [ ]:
if len(q_cart) > 0:
    display(animate_trajectory_comparison(robot, q_joint, q_cart))
else:
    print('Cartesian trajectory animation skipped because IK tracking failed.')


## 제출 전 체크

아래 항목을 스스로 확인해 보세요.

- `cubic_coefficients()`가 시작/끝 속도 0 조건을 만족하는가?
- via point를 넣었을 때 경로가 의도한 방향으로 바뀌는가?
- `estimate_via_velocities()`를 바꾸면 motion 느낌이 달라지는가?
- joint-space trajectory와 Cartesian-space trajectory의 차이를 설명할 수 있는가?
